# Lecture 1 · Guided practical: getting started and exploring Kigali Sentinel-2 data

**Machine Learning for Earth Observation · Year 3 · 45–60 minutes · work in pairs**

This first practical has two aims:
1. **Get started** with the tools used in every practical: Jupyter notebooks, the course conda environment, and basic Python/NumPy.
2. **Explore real EO data**: three Sentinel-2 Level-2A images of Kigali (21 January, 13 May and 3 September 2026).

| Part | What you do | Learning objectives | Time |
|---|---|---|---|
| 0 | Jupyter basics, environment check, a Python/NumPy warm-up | — | 10 min |
| 1 | Open the files: shape, data type, pixel size, extent, value ranges | raster data (LO3), quality (LO6) | 6 min |
| 2 | Band stack; true- and false-colour composites | multiband raster (LO3) | 6 min |
| 3 | Spectral signatures of five surface types | spectral characteristics (LO4) | 6 min |
| 4 | Degrade the spatial resolution: mixed pixels | resolution (LO2, LO5) | 6 min |
| 5 | Re-quantise to 8 bits: radiometric resolution | resolution (LO2) | 4 min |
| 6 | Compare three dates; mask clouds and no-data with the SCL | temporal resolution, quality (LO2, LO6) | 6 min |
| 7 | Build a small sample table: features x and label y | samples, features, labels (LO7) | 5 min |

Complete the cells marked **`# TODO`** (replace `...`) and write your answers in the **Your answer** boxes.

## Part 0 · Getting started

### 0.1 How a notebook works
* A notebook is a list of **cells**. *Markdown* cells (like this one) contain text; *code* cells contain Python.
* Run the selected cell with **Shift + Enter** (runs it and moves to the next cell). Ctrl + Enter runs it and stays.
* Cells share one Python session (the **kernel**). Variables created in one cell are available in later cells, **in the order you ran them**, not the order on the page.
* If something behaves strangely: *Kernel → Restart Kernel and Run All Cells…*
* `[*]` next to a cell means it is still running; a number, e.g. `[3]`, shows the order in which cells were run.
* Save often (Ctrl + S). Add your own notes in new Markdown cells (press **M** in command mode to turn a cell into Markdown).

### 0.2 Before you start (in a terminal / Anaconda Prompt)
```
conda activate ml4eo          # the course environment
cd path/to/ml4eo/L1_Practical # ml4eo/ also holds ml4eo_data.py and data/
jupyter lab                   # or: jupyter notebook
```
Check the kernel name at the top right of the notebook: it should be the `ml4eo` environment (e.g. *Python 3 (ml4eo)*).

### 0.3 Check your environment
Run the next cell. Every line should say **OK**. If a package is missing, see `GETTING_STARTED.md`.

In [ ]:
import sys, os, importlib

print("Python", sys.version.split()[0], "|", sys.executable)
print("Working directory:", os.getcwd())
for pkg in ["numpy", "pandas", "matplotlib", "scipy", "sklearn", "skimage"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  OK   {pkg:12s} {m.__version__}")
    except ImportError:
        print(f"  MISSING  {pkg}  ->  conda install -n ml4eo {pkg.replace('sklearn', 'scikit-learn').replace('skimage', 'scikit-image')}")

In [ ]:
# course data: the GeoTIFFs in ml4eo/data, read with the shared module ml4eo/ml4eo_data.py
sys.path.append("..")                       # ml4eo_data.py sits one folder up, in ml4eo/
try:
    import ml4eo_data as ml
    files = ml.list_files()
    print(f"  OK   data folder: {ml.DATA_DIR}")
    for date, n in files.groupby("date").size().items():
        print(f"       {date}: {n} files")           # expect 16 per date
    csv = ml.DATA_DIR / "kigali_training_polygons.csv"
    print(f"  {'OK  ' if csv.exists() else 'MISSING'} {csv.name}")
except (ImportError, FileNotFoundError) as e:
    print("  MISSING ", e)

### 0.4 A five-minute Python / NumPy warm-up
EO images are handled as **NumPy arrays**: grids of numbers with a `shape` and a `dtype`. Run the cell and read the comments.

In [ ]:
import numpy as np

a = np.array([[0.05, 0.08, 0.30],
              [0.04, 0.25, 0.32]])           # a tiny 2 x 3 'image' of reflectance values
print("shape:", a.shape, "| dtype:", a.dtype)  # (rows, columns)
print("row 0, column 2:", a[0, 2])            # indexing: [row, column], counting from 0
print("first row:", a[0, :])                  # slicing: ':' means 'all'
print("mean:", a.mean().round(3), "| max:", a.max())
print("values > 0.2:", a > 0.2)                # a Boolean 'mask' of the same shape
print("mean of those:", a[a > 0.2].mean().round(3))

red = np.array([0.05, 0.10, 0.04]); nir = np.array([0.30, 0.12, 0.03])
ndvi = (nir - red) / (nir + red)               # arithmetic works element by element
print("NDVI:", ndvi.round(2))

**Try it:** in the empty cell below, create an array `b` of shape (3, 4) filled with the value 0.1 (`np.full`), then print its shape and the sum of all its values.

In [ ]:
# TODO: an array b of shape (3, 4) filled with 0.1; print its shape and the sum of its values
b = ...
print(b.shape, b.sum().round(2))

## Step 1 · Open the files and inspect them  *(LO3, LO6)*

The Kigali images are **GeoTIFFs** exported from the Copernicus Browser: one file per band and date, in `ml4eo/data/`. All three dates were exported with the **same box and settings**: 20 m pixels in UTM zone 36S (EPSG:32736). All practicals read them with the course module `ml4eo_data` (Lecture 3 looks inside it).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

from ml4eo_data import list_files, read_tif, load_scene, load_polygons, grid, S2_BANDS

files = list_files()                     # one GeoTIFF per band and date
for date, grp in files.groupby("date"):
    print(date, ":", ", ".join(grp.layer))

# open three raw files with different NATIVE resolutions: 10 m (B04), 20 m (B05), 60 m (B01)
raw = {}
for b in ["B04", "B05", "B01"]:
    raw[b], g = read_tif(ml.DATA_DIR / files.query("date == '2026-09-03' and layer == @b").file.iloc[0])
    print(f"{b}: shape {raw[b].shape}, dtype {raw[b].dtype}, pixel {g['dx']:.2f} x {g['dy']:.2f} m, "
          f"upper-left corner ({g['ulx']:.0f}, {g['uly']:.0f}) m, {g['crs']}")

# zoom on 1.5 km x 1.5 km: the 60 m band looks blurred although it is stored with 20 m pixels
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (b, native) in zip(ax, [("B04", 10), ("B05", 20), ("B01", 60)]):
    a.imshow(raw[b][700:775, 600:675], cmap="gray"); a.set_title(f"{b}: native {native} m, stored at 20 m")
    a.set_xticks([]); a.set_yticks([])
plt.show()

All bands have the same shape, pixel size and corner: they are on **one grid**, so pixel (row, col) is the same place on the ground in every band and on every date. That was a choice made at export: Copernicus Browser **resampled** every band to 20 m. The 10 m bands lost detail, and the 60 m bands (B01, B09) were interpolated to 20 m: they look smooth and blurred, and they contain no more detail than before. The *stored* pixel size is not always the *real* resolution.

The files store reflectance (0–1). Many archives (for example the original Sentinel-2 SAFE products and Google Earth Engine) deliver integer **digital numbers (DN) = reflectance × 10,000** instead, so we ask for DN here and practise the conversion. **DN 0 means no data.**

In [ ]:
sep = load_scene("sep", scale="dn")   # dict: one 2-D array per band + "SCL"
BANDS = S2_BANDS
dn = np.stack([sep[b] for b in BANDS])   # digital numbers (DN), 0 = no data
g = grid()                              # the pixel grid: corner, pixel size, CRS
n_bands, H, W = dn.shape
print("shape (bands, rows, cols):", dn.shape, "| dtype:", dn.dtype)
print("DN range (valid pixels):", dn[dn > 0].min(), "-", dn.max(), "| no-data values (DN 0):", int((dn == 0).sum()))
print(f"memory: {dn.nbytes / 1e6:.1f} MB as uint16")

# UTM coordinates are in metres, so pixel size and extent can be read directly
px_x, px_y = g["dx"], g["dy"]
print(f"pixel size {px_x:.2f} m x {px_y:.2f} m; extent {W * px_x / 1000:.1f} km x {H * px_y / 1000:.1f} km; CRS {g['crs']}")

# TODO: convert DN to reflectance (reflectance = DN / 10000); use float32 and convert BEFORE any arithmetic
refl = ...
refl[dn == 0] = np.nan                   # 0 = no data
s = {b: refl[i] for i, b in enumerate(BANDS)}
print("reflectance B08: min %.3f, median %.3f, max %.3f" % (np.nanmin(s["B08"]), np.nanmedian(s["B08"]), np.nanmax(s["B08"])))
print("pixels with reflectance > 1 (any band):", int(np.nansum(refl > 1)))

**Question 1.** What are the dimensions, data type, pixel size and extent of the September stack? How does the pixel size compare with the native Sentinel-2 resolutions (10, 20, 60 m), and why? What does the value > 1 tell you?

*Your answer:*



## Step 2 · Band stack and colour composites  *(LO3)*

A colour image uses three bands as red, green and blue. We stretch each band between its 2nd and 98th percentile so that the image is not too dark.

In [ ]:
def stretch(a, lo=2, hi=98):
    l, h = np.nanpercentile(a, [lo, hi])
    return np.clip((a - l) / (h - l), 0, 1)

def composite(r, g, b):
    rgb = np.dstack([stretch(r), stretch(g), stretch(b)])
    rgb[np.isnan(rgb)] = 1                  # no data shown white
    return rgb

true_colour = composite(s["B04"], s["B03"], s["B02"])    # red, green, blue
# TODO: a standard false-colour composite: NIR (B08) as red, red (B04) as green, green (B03) as blue
false_colour = ...
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(true_colour); ax[0].set_title("True colour (B04, B03, B02)")
ax[1].imshow(false_colour); ax[1].set_title("False colour (B08, B04, B03)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()

**Question 2.** In the false-colour image, what colour are vegetation, water and the city? Why is vegetation so much brighter in this composite than in true colour?

*Your answer:*



## Step 3 · Spectral signatures of five surface types  *(LO4)*

`kigali_training_polygons.csv` lists 1,225 pixels (row, column) inside 49 polygons of five classes. We read the 12 reflectances at these pixels and average them per class.

In [ ]:
poly = load_polygons()
print(poly.head(3)); print(poly["class"].value_counts())
r, c = poly["row"].values, poly["col"].values
for b in BANDS:
    poly[b] = s[b][r, c]                      # the reflectance of band b at every sample pixel

WAVELENGTH = [443, 490, 560, 665, 705, 740, 783, 842, 865, 945, 1610, 2190]   # nm, band centres
CC = {"water": "#2A7FA8", "dense vegetation": "#2E6B3F", "sparse vegetation": "#A8C66C", "built-up": "#B5503C", "bare soil": "#9A8468"}
# TODO: mean and standard deviation of each band per class (hint: poly.groupby("class")[BANDS].mean())
means = ...
sds = ...
plt.figure(figsize=(9, 4.5))
for cls in CC:
    plt.plot(WAVELENGTH, means.loc[cls], "o-", color=CC[cls], label=cls)
    plt.fill_between(WAVELENGTH, means.loc[cls] - sds.loc[cls], means.loc[cls] + sds.loc[cls], color=CC[cls], alpha=0.15)
plt.xlabel("wavelength (nm)"); plt.ylabel("surface reflectance"); plt.legend(); plt.grid(alpha=0.3); plt.show()
means.round(3)

**Question 3.** Which bands best separate (a) water from everything else, (b) vegetation from non-vegetation, (c) dense from sparse vegetation? Which pair of classes is hardest to separate, and why?

*Your answer:*



## Step 4 · Spatial resolution: what happens with bigger pixels?  *(LO2, LO5)*

We simulate coarser sensors by averaging blocks of *f × f* pixels (f = 5 → 100 m, 15 → 300 m, 50 → 1 km). For each coarse pixel we also check how *pure* it is using the scene classification layer (SCL: 4 = vegetation, 5 = not vegetated, 6 = water).

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")   # coarse pixels in the no-data frame
scl = sep["SCL"]
ndvi = (s["B08"] - s["B04"]) / (s["B08"] + s["B04"])

def blocks(a, f):
    """Cut array a into non-overlapping f x f blocks: result shape (rows/f, cols/f, f*f)."""
    h, w = (a.shape[0] // f) * f, (a.shape[1] // f) * f
    return a[:h, :w].reshape(h // f, f, w // f, f).transpose(0, 2, 1, 3).reshape(h // f, w // f, f * f)

rows, coarse = [], {1: ndvi}
for f in [5, 15, 50]:
    b = blocks(scl, f)
    share = np.stack([(b == k).mean(axis=-1) for k in [4, 5, 6]])      # share of each SCL class per coarse pixel
    purity = share.max(axis=0)
    # TODO: the coarse NDVI = mean of each f x f block (use blocks() and np.nanmean over the last axis)
    coarse[f] = ...
    rows.append({"pixel size (m)": int(round(f * px_x, -1)), "coarse pixels": purity.size,
                 "% ≥ 90% one class": 100 * (purity >= 0.9).mean(), "% strongly mixed (< 60%)": 100 * (purity < 0.6).mean(),
                 "% water (majority)": 100 * (share[2] > 0.5).mean()})
print(f"water at the original 20 m: {100 * (scl == 6).mean():.2f}% of pixels")

fig, ax = plt.subplots(1, 4, figsize=(17, 3.5))
zoom = (slice(450, 900), slice(300, 950))          # central Kigali
for a, f in zip(ax, [1, 5, 15, 50]):
    z = coarse[f][zoom[0].start // f:zoom[0].stop // f, zoom[1].start // f:zoom[1].stop // f]
    a.imshow(z, cmap="YlGn", vmin=0, vmax=0.7, interpolation="nearest"); a.set_title(f"NDVI, {int(round(f * px_x, -1))} m"); a.set_xticks([]); a.set_yticks([])
plt.show()
pd.DataFrame(rows).round(1)

**Question 4.** What happens to the share of pure pixels and to the small water bodies as pixels get larger? For which of these tasks would 1 km pixels be acceptable: (a) mapping Kigali's built-up area, (b) monitoring seasonal greenness of Rwanda as a whole?

*Your answer:*



## Step 5 · Radiometric resolution: 8 bits vs the original  *(LO2)*

Sentinel-2 records 12-bit data (4,096 levels). What if we stored reflectance with only 8 bits (256 levels), like a JPEG screenshot?

In [ ]:
def quantise(a, bits):
    levels = 2 ** bits - 1
    # TODO: clip a to 0-1, scale to 0..levels, round to whole numbers, and scale back to 0-1
    return ...

water, veg = (scl == 6), (scl == 4)
rows = []
for bits in [8, 10, 12]:
    r8, n8 = quantise(s["B04"], bits), quantise(s["B08"], bits)
    ndvi_q = (n8 - r8) / (n8 + r8)
    err = ndvi_q - ndvi
    rows.append({"bits": bits, "levels": 2 ** bits, "distinct red values over water": len(np.unique(r8[water])),
                 "NDVI RMSE water": np.sqrt(np.nanmean(err[water] ** 2)), "NDVI RMSE vegetation": np.sqrt(np.nanmean(err[veg] ** 2))})
print("distinct red values over water in the original data:", len(np.unique(s["B04"][water])))
pd.DataFrame(rows).round(4)

**Question 5.** Why does 8-bit storage hurt NDVI over water much more than over vegetation? What does this imply for using screenshots or JPEG images for analysis?

*Your answer:*



## Step 6 · Three dates and data quality  *(LO2, LO6)*

21 January and 13 May 2026 were exported with the same box and settings as September, so all three dates share one grid and can be compared pixel by pixel. We load only the four bands we need. SCL codes: 0 no data, 3 cloud shadow, 4 vegetation, 5 not vegetated, 6 water, 8/9 cloud, 10 thin cirrus.

In [ ]:
dates = {"21 Jan 2026": "jan", "13 May 2026": "may", "3 Sept 2026": "sep"}
SCL_NAMES = {0: "no data", 2: "dark area", 3: "cloud shadow", 4: "vegetation", 5: "not vegetated", 6: "water", 7: "unclassified", 8: "cloud (medium)", 9: "cloud (high)", 10: "thin cirrus"}

scl_d, rgb_d, ndvi_d = {}, {}, {}
for label, key in dates.items():
    if key == "sep":
        b = {k: s[k] for k in ["B02", "B03", "B04", "B08"]}; sc = scl
    else:
        b = load_scene(key, ["B02", "B03", "B04", "B08"])   # reflectance (0-1), NaN = no data
        sc = b["SCL"]
    scl_d[label] = sc
    rgb_d[label] = composite(b["B04"], b["B03"], b["B02"])
    ndvi_d[label] = (b["B08"] - b["B04"]) / (b["B08"] + b["B04"])

shares = pd.DataFrame({label: pd.Series(sc.ravel()).map(SCL_NAMES).value_counts(normalize=True) * 100 for label, sc in scl_d.items()}).fillna(0).round(2)
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, (label, im) in zip(ax, rgb_d.items()):
    a.imshow(im); a.set_title(label); a.set_xticks([]); a.set_yticks([])
plt.show()
shares

Now build a **validity mask** for each date: keep only vegetation, not vegetated and water (SCL 4, 5, 6), and compare the median NDVI of pixels that are valid on all three dates.

In [ ]:
# TODO: for each date, a Boolean mask that is True where the SCL is 4, 5 or 6 (hint: np.isin) AND NDVI is finite
valid = {label: ... for label, sc in scl_d.items()}
common = valid["21 Jan 2026"] & valid["13 May 2026"] & valid["3 Sept 2026"]
print(f"pixels valid on all three dates: {100 * common.mean():.1f}% of the grid")
for label in dates:
    print(f"  {label}: {100 * valid[label].mean():5.1f}% valid | median NDVI (common pixels) = {np.nanmedian(ndvi_d[label][common]):.2f}")

**Question 6.** (a) Which data-quality problems do you see in the January image (look at the pictures and the SCL table)? (b) What share of the pixels is valid on all three dates, and what removes the rest? (c) How and why does the median NDVI change between the dates, and what does that mean for a classifier trained on the September image?

*Your answer:*



## Step 7 · From pixels to an ML table: samples, features and labels  *(LO7)*

Each **sample** is one pixel; its **features x** are band values (and NDVI); its **label y** is the class of its polygon. We build the table and save it; Lecture 2 uses this kind of table to train a model.

In [ ]:
FEATURES = ["B02", "B03", "B04", "B08", "B11", "B12"]
samples = poly[["obs_id", "group_id", "row", "col", "class"]].copy()
for f in FEATURES:
    # TODO: the value of band f at each sample's row and column
    samples[f] = ...
samples["NDVI"] = ndvi[samples["row"], samples["col"]]
samples["SCL"] = scl[samples["row"], samples["col"]]
print(samples.shape)
print(samples.head())
print("\nSamples per class:"); print(samples["class"].value_counts())
print("\nSCL code of the samples, per class:"); print(pd.crosstab(samples["class"], samples["SCL"]))
samples.to_csv("kigali_samples_L1.csv", index=False)
print("saved kigali_samples_L1.csv")

**Question 7.** In this table, what is one sample, what are the features and what is the label? The 1,225 pixels come from only 49 polygons: why might that matter? Compare the class labels with the SCL codes: where do they disagree, and why can both be 'right'?

*Your answer:*



## Reflection (submit in pairs)

1. Write down three facts about the Kigali data you would put in the *Data* section of a report (sensor, level, date, pixel size, extent, bands, quality).
2. Name one data-quality check from today you will always do before training a model, and why.
3. Which of the four resolutions limited today's analysis most? Explain with one result from this notebook.

*Your answer:*

